# Backtest BB Mean Reversion

Run the Bollinger Band Mean Reversion strategy against BTC-USD-PERP 1h data on a simulated Hyperliquid venue.

**Strategy logic:**
- **Entry:** Price touches lower BB band AND RSI < buy threshold → Long. Price touches upper BB band AND RSI > sell threshold → Short.
- **Exit:** Price reverts to middle BB band (SMA baseline).
- BB uses typical price `(high+low+close)/3` internally for band calculation.
- NT's RSI uses **0.0–1.0 scale** (not 0–100). RSI 30 = `0.30`.

**Sections:**
1. Setup — imports, catalog, engine configuration
2. Single run — fills, positions, account reports
3. Visualisation — price + BB bands + RSI panels, equity curve
4. Statistics — analyzer performance summary
5. Parameter sweeps — BB period/std grid, RSI threshold grid

**Prerequisites:** Run `scripts/fetch_hl_candles.py` first to populate `data/catalog/`.

In [ ]:
# ── Cell 1: Imports + shared config ────────────────────────────────────────
#
# All tuneable values live here. Change once, used everywhere below.

from decimal import Decimal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML

from nautilus_trader.analysis import create_tearsheet
from nautilus_trader.indicators import BollingerBands, RelativeStrengthIndex
from nautilus_trader.model.data import BarType
from nautilus_trader.model.identifiers import Venue
from nautilus_trader.persistence.catalog import ParquetDataCatalog

from src.backtesting import make_engine, run_sweep
from src.core import bar_type_str, with_venue_config
from src.strategies.bb_meanrev import BBMeanRev, BBMeanRevConfig

from charts import (
    plot_bb_meanrev, plot_equity_curve, print_summary_stats,
    plot_pnl_heatmap, generate_backtest_html,
)
from utils import make_instrument_id, save_notebook, save_notebook_html, save_tearsheet

# ── Shared config ────────────────────────────────────────────────────
EXCHANGE         = "HYPERLIQUID"
ASSET            = "BTC"
BAR_INTERVAL     = "1d"
STARTING_CAPITAL = 10_000
TRADE_SIZE       = 1000              # $1,000 USD per trade
LEVERAGE         = 20                # Backtesting leverage (margin_init = 1/20 = 5%)
SAVE_TEARSHEET   = False

# BB params
BB_PERIOD = 20
BB_STD    = 2.0

# RSI params (NT RSI uses 0.0-1.0 scale!)
RSI_PERIOD         = 14
RSI_BUY_THRESHOLD  = 0.30
RSI_SELL_THRESHOLD = 0.70

# Sweep grids — BB
BB_PERIODS = [15, 20, 25, 30]
BB_STDS    = [1.5, 2.0, 2.5, 3.0]

# Sweep grids — RSI
RSI_BUY_VALUES  = [0.20, 0.25, 0.30, 0.35]
RSI_SELL_VALUES = [0.65, 0.70, 0.75, 0.80]

# Standard
TRADE_NOTIONAL = Decimal(TRADE_SIZE)
INSTRUMENT_ID  = make_instrument_id(ASSET, EXCHANGE)
TRADING_PAIR   = INSTRUMENT_ID.split("-")[0]
BAR_TYPE_STR   = bar_type_str(INSTRUMENT_ID, BAR_INTERVAL)
VENUE          = Venue(EXCHANGE)

# Files
CATALOG_PATH = "../data/catalog"
RESULT_NAME  = f"BBMeanRev_{INSTRUMENT_ID}_{BAR_INTERVAL}_p{BB_PERIOD}_s{BB_STD}"

print("Imports OK")

In [ ]:
# ── Cell 2: Load data from catalog ────────────────────────────────────────
catalog    = ParquetDataCatalog(CATALOG_PATH)
instrument = catalog.instruments(instrument_ids=[INSTRUMENT_ID])[0]
bars       = catalog.bars(bar_types=[BAR_TYPE_STR])

# Override margins for backtesting (catalog instruments have raw/live margins)
instrument = with_venue_config(instrument, LEVERAGE)

# Settlement currency for PnL queries (USDC for HL, USDT for Binance)
CURRENCY = instrument.settlement_currency

print(f"Instrument : {instrument.id}")
print(f"Currency   : {CURRENCY}")
print(f"Leverage   : {LEVERAGE}x (margin_init={instrument.margin_init}, margin_maint={instrument.margin_maint})")
print(f"Bar count  : {len(bars):,}")
print(f"First bar  : {pd.Timestamp(bars[0].ts_event, unit='ns', tz='UTC')}")
print(f"Last bar   : {pd.Timestamp(bars[-1].ts_event, unit='ns', tz='UTC')}")

In [ ]:
# ── Cell 3: Configure engine + venue ──────────────────────────────────────
engine = make_engine(VENUE, instrument, bars, STARTING_CAPITAL)
print("Engine configured.")

In [ ]:
# ── Cell 4: Add BBMeanRev strategy + run ────────────────────────────────
config = BBMeanRevConfig(
    instrument_id=instrument.id,
    bar_type=BarType.from_str(BAR_TYPE_STR),
    trade_notional=TRADE_NOTIONAL,
    bb_period=BB_PERIOD,
    bb_std=BB_STD,
    rsi_period=RSI_PERIOD,
    rsi_buy_threshold=RSI_BUY_THRESHOLD,
    rsi_sell_threshold=RSI_SELL_THRESHOLD,
)
strategy = BBMeanRev(config=config)
engine.add_strategy(strategy)
engine.run()
print("Backtest complete.")

In [ ]:
# ── Cell 5: Reports ───────────────────────────────────────────────────
fills_report     = engine.trader.generate_order_fills_report()
positions_report = engine.trader.generate_positions_report()
account_report   = engine.trader.generate_account_report(VENUE)

print(f"Order fills : {len(fills_report)}")
print(f"Positions   : {len(positions_report)}")
print()

print("=== Recent Fills ===")
display(fills_report.tail(10))

print("\n=== Recent Positions ===")
display(positions_report.tail(10))

In [ ]:
# ── Cell 6: Calculate statistics ──────────────────────────────────────────
#
# Must run before equity curve — analyzer.returns() is a getter that
# only has data after calculate_statistics() populates it.

account   = engine.cache.account_for_venue(VENUE)
positions = engine.cache.position_snapshots() + engine.cache.positions()
analyzer  = engine.portfolio.analyzer
analyzer.calculate_statistics(account, positions)
print(f"Stats calculated — {len(positions)} positions")

In [ ]:
# ── Cell 7: HTML Tearsheet ────────────────────────────────────────────────
html = create_tearsheet(
    engine,
    output_path=None,
    title=f"BB({BB_PERIOD}, {BB_STD}) + RSI({RSI_PERIOD}) | {INSTRUMENT_ID} {BAR_INTERVAL}",
)
display(HTML(html))

if SAVE_TEARSHEET:
    save_tearsheet(html, RESULT_NAME)

In [ ]:
# ── Cell 8: Plotly chart ───────────────────────────────────────────────────

fig = plot_bb_meanrev(
    bars, fills_report,
    bb_period=BB_PERIOD,
    bb_std=BB_STD,
    rsi_period=RSI_PERIOD,
    rsi_buy_threshold=RSI_BUY_THRESHOLD,
    rsi_sell_threshold=RSI_SELL_THRESHOLD,
    instrument_label=INSTRUMENT_ID,
    bar_label=BAR_INTERVAL,
)
fig.show(config=dict(
    modeBarButtonsToRemove=["autoScale2d", "lasso2d", "select2d"],
    displaylogo=False,
))

In [ ]:
# ── Cell 9: Equity curve ──────────────────────────────────────────────────
plot_equity_curve(
    analyzer, account_report,
    f"BB({BB_PERIOD}, {BB_STD}) + RSI({RSI_PERIOD})  {INSTRUMENT_ID} {BAR_INTERVAL}",
)

In [ ]:
# ── Cell 10: Summary statistics ───────────────────────────────────────────
print_summary_stats(analyzer, num_positions=len(positions), currency=CURRENCY)

In [ ]:
# ── Cell 11: Parameter sweep — BB period vs BB std ────────────────────
#
# Grid search over BB period and standard deviation combinations.
# RSI params are held at defaults.

def bb_meanrev_factory(eng, params):
    cfg = BBMeanRevConfig(
        instrument_id=instrument.id,
        bar_type=BarType.from_str(BAR_TYPE_STR),
        trade_notional=TRADE_NOTIONAL,
        bb_period=params.get("bb_period", BB_PERIOD),
        bb_std=params.get("bb_std", BB_STD),
        rsi_period=params.get("rsi_period", RSI_PERIOD),
        rsi_buy_threshold=params.get("rsi_buy", RSI_BUY_THRESHOLD),
        rsi_sell_threshold=params.get("rsi_sell", RSI_SELL_THRESHOLD),
    )
    eng.add_strategy(BBMeanRev(cfg))

combos = [
    {"bb_period": p, "bb_std": s}
    for p in BB_PERIODS
    for s in BB_STDS
]

results_df = run_sweep(
    venue=VENUE, instrument=instrument, bars=bars,
    starting_capital=STARTING_CAPITAL,
    param_combos=combos,
    strategy_factory=bb_meanrev_factory,
    strategy_name="BBMeanRev",
    instrument_id=INSTRUMENT_ID,
    bar_interval=BAR_INTERVAL,
)

In [ ]:
# ── Cell 12: PnL heatmap (BB period vs BB std) ─────────────────────────
plot_pnl_heatmap(
    results_df, row_col="bb_std", col_col="bb_period",
    row_label="BB Std Multiplier", col_label="BB Period",
    title=f"Total PnL ({CURRENCY}) by BB Parameters",
)

In [ ]:
# ── Cell 13: RSI sensitivity sweep ────────────────────────────────────────
#
# Fix BB at the best combo from the previous sweep, vary RSI thresholds.

best_row = results_df.loc[results_df["total_pnl"].idxmax()]
best_bb_period = int(best_row["bb_period"])
best_bb_std = float(best_row["bb_std"])
print(f"Best BB params: period={best_bb_period}, std={best_bb_std} (PnL={best_row['total_pnl']:,.2f})")

combos = [
    {"rsi_buy": buy, "rsi_sell": sell, "bb_period": best_bb_period, "bb_std": best_bb_std}
    for sell in RSI_SELL_VALUES
    for buy in RSI_BUY_VALUES
    if buy < sell
]

rsi_df = run_sweep(
    venue=VENUE, instrument=instrument, bars=bars,
    starting_capital=STARTING_CAPITAL,
    param_combos=combos,
    strategy_factory=bb_meanrev_factory,
    strategy_name="BBMeanRev-RSI-sensitivity",
    instrument_id=INSTRUMENT_ID,
    bar_interval=BAR_INTERVAL,
)

plot_pnl_heatmap(
    rsi_df, row_col="rsi_sell", col_col="rsi_buy",
    row_label="RSI Sell Threshold", col_label="RSI Buy Threshold",
    title=f"Total PnL ({CURRENCY}) by RSI Thresholds — BB({best_bb_period}, {best_bb_std})",
)

In [ ]:
# ── Cell 14: TradingView Interactive Backtest ──────────────────────────
#
# Uses SMA overlay (the middle BB band) since generate_backtest_html
# doesn't support BB bands natively. The Plotly chart above shows full bands.

path = generate_backtest_html(
    bars, fills_report, positions_report,
    fast_period=BB_PERIOD,
    slow_period=BB_PERIOD,
    ma_type="SMA",
    instrument_label=INSTRUMENT_ID,
    bar_label=BAR_INTERVAL,
    starting_capital=STARTING_CAPITAL,
    result_filename=RESULT_NAME,
)
# Then just open the file in your browser

In [ ]:
# ── Cell 15: Save notebook snapshot ──────────────────────────────────────
#
# Save the notebook (Ctrl+S) before running this cell.

#save_notebook("backtest_bb_meanrev.ipynb", RESULT_NAME)
#save_notebook_html("backtest_bb_meanrev.ipynb", RESULT_NAME)

In [ ]:
# ── Cell 16: Cleanup ──────────────────────────────────────────────────
engine.dispose()
print("Engine disposed.")